<!-- SpatialScore: Towards Unified Evaluation for Multimodal Spatial Understanding -->
# SpatialScore：迈向多模态空间理解的统一评估 

## 摘要

现有对多模态大语言模型（MLLMs）在空间智能方面的评估通常是零散且范围有限的。在这项工作中，我们旨在对现代 MLLMs 的空间理解能力进行整体评估，并提出补充性的数据驱动和基于代理的解决方案。具体而言，我们做出了以下贡献：

(i) 我们介绍了 SpatialScore，据我们所知，这是迄今为止最全面、最多样化的多模态空间智能基准。 它涵盖了多种视觉数据类型、输入模态和问答格式，包含约 5K 个经过人工验证的样本，涉及 30 种不同任务；
(ii) 使用 SpatialScore，我们广泛评估了 40 个有代表性的 MLLM，揭示了持续存在的挑战以及当前模型与人类水平空间智能之间的巨大差距；
(iii) 为提升模型能力，我们构建了 SpatialCorpus，这是一个包含 331K 个多模态问答样本的大规模训练资源，支持在空间推理任务上进行微调，并显著提升了现有模型（例如 Qwen3-VL）的性能；
(iv) 为补充这一数据驱动路线，我们开发了 SpatialAgent，这是一个配备 12 个专用空间感知工具的多智能体系统，支持 Plan-Execute 和 ReAct 推理，能够在无需额外模型训练的情况下显著提升空间推理能力。大量的实验和深入分析证明了我们的基准、语料库和智能体框架的有效性。

我们期望这些资源能为 MLLM 向人类水平空间智能的迈进提供坚实的基础。 所有数据、代码和模型将向研究社区开放。

## 1 引言

- 多模态大语言模型（MLLMs）虽在语义问答、数学推理等领域表现出色，但空间智能相关进展零散。空间推理能力对具身智能、自主导航等现实应用至关重要，而传统计算机视觉的空间感知方法缺乏与语言的紧密整合及统一评估协议，因此语义理解与空间感知的融合成为前沿方向，核心问题是探究现有 MLLMs 的空间智能（含感知与理解）水平。
- 近期相关研究尚处萌芽阶段，存在两大关键局限：
  - （1）任务简化，现有基准聚焦浅层空间查询，忽视严格视觉几何感知；
  - （2）评估范围狭窄，评估碎片化，依赖简单问题、单模态输入或孤立技能，无法全面衡量空间智能。

为应对上述挑战，作者改造 3D 数据集并整合 23 个现有数据集的空间相关样本，构建了多样化、全面的空间理解基准 SpatialScore。该基准含约 5K 人工验证高质量样本，覆盖 30 种空间推理任务、多种数据类型（真实 / 模拟 / AIGC）、模态（图像 / 视频）及问题格式（判断 / 多选 / 开放式问答）。对 40 个代表性 MLLM 的评估表明，其空间智能仍面临挑战，与人类表现差距显著。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/teaser.png" />
    <span style=" font-size: 12px; color: black;"><strong>图1</strong>：概述。（a）SpatialScore 中不同类别的代表性示例，通过问答（判断、多选和开放式问答）全面评估空间智能能力；（b）最先进模型在 SpatialScore 上的表现与人类表现的比较。</span>
</div>

为增强 MLLMs 的空间推理能力，我们探索两条互补途径：
- 一是构建含 331K 多模态空间问答样本的 SpatialCorpus 训练资源，通过监督微调提升模型性能；
- 二是提出 SpatialAgent 智能体框架，协调 12 个空间感知工具，支持 Plan-Execute（分层任务分解 + 顺序工具调用）和 ReAct（交替推理与行动）两种推理范式，以无训练方式增强现成 MLLMs 的空间理解能力。

本文后续结构：第 2 节详述 SpatialScore 基准构建及模型评估；第 3 节介绍空间智能提升策略（SpatialCorpus 与 SpatialAgent）；第 4 节说明实验方案并呈现实验结果；第 5 节回顾相关文献；第 6 节总结核心见解与贡献。本研究建立了迄今最全面多样的空间智能基准，以期推动 MLLMs 相关领域进展。

## 2 SpatialScore

- 2.1节介绍SpatialScore基准的构建过程。
- 2.2节进行详细的统计分析和数据讨论
- 2.3节评估了40个代表性MLLM在SpatialScore上的表现。

### 2.1 数据集构建

为全面评估多模态大语言模型（MLLMs）的空间理解能力，我们构建了SpatialScore 基准，据我们所知，这是目前覆盖最全面、类型最多样的空间智能评测基准。该基准整合了两部分数据：
- 一是基于现有 3D 标注改造的全新问答数据，
- 二是来自 23 个公开数据集的空间相关样本，具体构建流程如下。

__3D 数据改造__

现有 MLLM 基准中，针对 3D 视觉几何感知（如相机姿态、点追踪、深度估计）的问答数据十分匮乏。为此，我们设计了一套可扩展、可控的流程，利用精准 3D 标注（如深度、3D 包围盒）生成高质量问答对。流程具体为：
从 5 个含精确 3D 标注的数据集（ScanNet++、Omni3D 等）中随机抽取 500 个场景；
结合预设问题模板与大语言模型改写（如调用 DeepSeek-v3 将基础问题转化为多样化表述），构建开放式问答对；

为支持量化评估，将部分开放式问答对转化为判断（是 / 否）和多选格式，并通过三类策略生成合理干扰项：在合理数值范围内从同 / 异场景抽取同类标注、对真实值施加小幅扰动、借助 DeepSeek-v3 合成具有迷惑性的有效干扰项。
此步骤最终得到 2300 条涵盖判断、多选、开放三种格式的高质量问答样本。

__数据整合与筛选__

我们进一步整合了来自三类现有数据集的空间智能评测样本：认知心理学类（SRBench 等）、2D/3D 空间关系与距离推理类（SpatialSense 等）、通用问答基准中的空间相关子集（CV-Bench 等）。
整合后共得到 63857 条候选样本，随后通过以下步骤筛选优化：
采用高性能大语言模型（GPT-OSS-120B）过滤无需视觉信息即可回答的问题，候选样本缩减至 40238 条；
经人工严谨核验与题型重分类，最终筛选出5025 条高质量、分布均衡的样本（含 1091 条全新样本），覆盖 30 项任务，构成 SpatialScore 基准。
这些任务按特性被划分为 10 个直观类别：心理模拟、计数、深度估计、物体距离、物体运动、相机位姿与运动、时序推理、视角推理、物体尺寸、物体定位。

### 2.2 统计与讨论

图 2 (b) 展示SpatialScore与SpatialCorpus的类别特异性数据分布（详见第 3.2 节）。表 1 对比代表性空间智能基准，以验证本文精选数据的全面性与多样性。该基准涵盖三类数据（真实世界、模拟、AIGC）、三种输入模态（单图像、多帧序列、视频）及三种题型（多选、判断、开放式问答），更多细节见附录 A。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/benchmark_comparison.png" />
    <span style=" font-size: 12px; color: black;"><strong>表1</strong>：与现有空间智能基准的比较。此处，Real 和 AIGC 分别表示真实世界样本和由视觉生成模型生成的数据。所考虑的输入模态包括单图像、多图像序列和视频。</span>
</div>

### 2.3 在 SpatialScore 上的代表性模型比较

为全面评估空间推理能力，我们在 **40 个不同规模的代表性多模态大语言模型（MLLM）** 上，基于所提的 SpatialScore 开展了大量实验，涵盖通用模型（如 InternVL 系列、Qwen 系列、Kimi-VL、LLaVA-1.5、LLaVA-OneVision、LLaMA-3.2V、LLaMA-3.2V-CoT）、空间理解专用微调模型（如 SpaceQwen2.5VL、SpaceThinker、SpaceLLaVA、SpaceR）及闭源模型（如 GPT-5、Gemini、Claude-4.5-Sonnet）。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/quantitative_results.png" />
    <span style=" font-size: 12px; color: black;"><strong>表2</strong>：SpatialScore 结果。Mental., Count., Depth., Obj-Dist., Obj-Mo., Camera., Temp-Rea., View-Rea., Obj-Size., Obj-Loc.分别指代心理动画、计数、深度估计、物体距离、物体运动、相机姿态与运动、时间推理、视图推理、物体大小和物体定位。每组中的最佳和次佳结果以加粗和下划线标出。</span>
</div>

表2报告代表性多模态大语言模型（MLLMs）在**SpatialScore**基准的量化结果，可得四项关键结论：
1.  **整体性能**：GPT-5（60.12）表现最优，开源模型中Qwen3-VL-235B-A22B（56.63）性能最佳，已大幅缩小与闭源系统的差距，但当前最优模型与人类空间理解水平（86.60）仍存26.48的显著差距；
2.  **模型规模与性能**：大模型通常性能更优，InternVL、Qwen-VL系列均体现此趋势，印证更强的内在推理能力可转化为空间智能的提升；
3.  **现有微调的局限性**：空间专用微调模型（如SpaceThinker、SpaceR）性能提升有限，部分甚至不及基础模型（如Qwen2.5-VL），其泛化能力不足突显**SpatialScore**的多样性与难度，也暴露当前空间理解微调策略及数据集的缺陷；
4.  **当前模型的局限性**：部分模型在心理动画、物体定位等基础任务上接近人类水平，但在视图推理、相机姿态、运动分析及真实世界3D感知等任务中仍面临显著挑战，暴露出当代MLLMs在真实3D理解方面的明显短板。

## 3 方法

本节探讨提升 MLLMs 空间理解能力的两类互补方法（数据驱动法、智能体中心法）。3.1 节界定问题范围，3.2 节介绍基于 SpatialCorpus 的监督微调方案，3.3 节描述 SpatialAgent 多智能体系统，3.4 节详述其集成的空间感知工具集。

### 3.1 问题公式化


本文采用问答范式评估并提升空间理解能力。给定文本问题 $\boldsymbol{q}$ 与视觉输入 $\boldsymbol{v}$（单图像、多帧序列或视频），基于多模态大语言模型（MLLM）的基础问答过程可表示为：
$$\boldsymbol{r} = \Phi(\boldsymbol{q}, \boldsymbol{v})$$
其中 $\Phi(\cdot)$ 代表 MLLM，$\boldsymbol{r}$ 为自由文本回答。
在数据驱动方案中，模型通过在 **SpatialCorpus** 上监督微调实现能力提升；推理形式保持不变，仅将 $\Phi$ 替换为参数更新后的微调模型 $\hat{\Phi}$。

与之并行的另一方案，保留预训练 MLLM 主体，在推理阶段引入 **SpatialAgent**（$\mathcal{A}$）智能体框架进行增强。该多智能体系统可在推理过程中调度外部空间工具，核心流程表示为：
$$\boldsymbol{r} = \mathcal{A}\bigl(\Phi(\boldsymbol{q}, \boldsymbol{v}), \mathcal{T}\bigr)$$
其中 $\mathcal{T}=\{\boldsymbol{t}_1,\boldsymbol{t}_2,\dots,\boldsymbol{t}_n\}$ 为专用空间感知工具集（各 $\boldsymbol{t}_i$ 对应独立工具），详见第 3.4 节。

上述两种方案形成互补：一是通过数据直接强化骨干模型，二是固定模型主体并借助工具推理实现能力增强。

### 3.2 有监督微调

**核心方法**
提升空间理解能力的基础且有效手段是**基于领域专用数据的有监督微调（SFT）**。训练样本为多模态三元组 `(视觉场景𝐯, 空间相关问题𝐪, 真实答案𝐫)`，训练目标是最小化模型生成回答与真实答案的差异(MLLM)，优化模型在位置、距离、相机变换等维度的推理能力，最终得到专精于空间智能的微调模型 $\hat{\Phi}$。

**数据集构建**
为支撑该方法，本文构建自动化数据筛选流程，打造了**空间数据集（SpatialCorpus）**：
1.  整合真实与模拟环境数据，涵盖单帧/多帧输入，支持选择、判断、开放式问答等多样格式；
2.  沿用2.1节的数据复用策略构建训练集，确保与测试集无重叠；
3.  剔除仅含通用类别标注的CA-1M数据集；
4.  采用规则模板替代大模型改写以降低成本，引入模拟器生成的空间地图、2D/3D旋转序列等合成数据，增强模型的动态空间想象能力。

**实验结果**
最终的SpatialCorpus包含**7大类16个任务的33.1万组问答对**。基于此数据集，采用交叉熵损失对主流开源模型Qwen3-VL进行微调，在各类空间推理任务中均取得稳定性能提升（详见4.2节）。

### 3.3 SpatialAgent智能体

**研究动机**
有监督微调（SFT）存在计算成本高昂、易过拟合及可能**灾难性遗忘通用能力**等问题，因此本文构建**无训练替代方案SpatialAgent**。该方法通过精心设计的提示词，引导模型核心$\Phi$划分功能角色，并执行**规划-执行（PE）** 与**ReAct**两种推理范式，以提升空间理解能力。

**PE范式（Plan-Execute Paradigm）**
如图3(a)所示，SpatialAgent由规划器、执行器和总结器构成（记为$\mathcal{A}_{\text{PE}}=\{\Phi_{\text{plan}},\Phi_{\text{exec}},\Phi_{\text{sum}}\}$），采用**串行前馈流程**生成最终回答$\boldsymbol{r}_{\text{PE}}$：
1. 规划器基于视觉输入$\boldsymbol{v}$、问题$\boldsymbol{q}$及工具箱$\mathcal{T}$的详细说明，生成含$k$个步骤的工具调用计划$\boldsymbol{p}$，每个步骤指定工具$\boldsymbol{t}_i$及参数$\text{args}_i$；
$$\boldsymbol{p} = \Phi_{\text{plan}}(\boldsymbol{q}, \boldsymbol{v}; \mathcal{T}) = \{(\boldsymbol{t}_1, \text{args}_1), (\boldsymbol{t}_2, \text{args}_2), \dots, (\boldsymbol{t}_k, \text{args}_k)\}$$
2. 执行器按序执行计划，输出各步骤结果集合$\mathcal{Y}$；
$$\mathcal{Y} = \{\boldsymbol{y}_1, \boldsymbol{y}_2, \dots, \boldsymbol{y}_k\} = \Phi_{\text{exec}}(\boldsymbol{p})$$
3. 总结器结合工具输出$\mathcal{Y}$与原始输入，推理得到最终回答。
$$\boldsymbol{r}_{\text{PE}} = \Phi_{\text{sum}}(\boldsymbol{q}, \boldsymbol{v}, \mathcal{Y})$$

**ReAct范式（Reasoning and Acting Paradigm）**
如图3(b)所示，该范式采用**交替迭代推理流程**，SpatialAgent由观测器、执行器和总结器构成（记为$\mathcal{A}_{\text{ReAct}}=\{\Phi_{\text{obs}},\Phi_{\text{exec}},\Phi_{\text{sum}}\}$），并配备**记忆模块$\mathcal{M}$** 记录观测器与执行器的交互历史：
1.  每一步$i$的记忆状态$\mathcal{M}_i$存储过往观测决策$\boldsymbol{o}$与执行结果$\boldsymbol{y}$；
$$\mathcal{M}_i = \{m_1, m_2, \dots, m_{i-1}\} = \{(\boldsymbol{o}_1, \boldsymbol{y}_1), (\boldsymbol{o}_2, \boldsymbol{y}_2), \dots, (\boldsymbol{o}_{i-1}, \boldsymbol{y}_{i-1})\}, \quad \text{with } \mathcal{M}_1 = \varnothing$$
2.  观测器基于输入$\{\boldsymbol{q},\boldsymbol{v}\}$和交互历史$\mathcal{M}_i$生成下一步动作$\boldsymbol{o}_i$，执行器据此完成操作；
$$\boldsymbol{o}_i = \Phi_{\text{obs}}(\mathcal{M}_i, \boldsymbol{q}, \boldsymbol{v}); \boldsymbol{y}_i = \Phi_{\text{exec}}(\boldsymbol{o}_i)$$
3.  迭代直至观测器输出终止指令，总结器整合记忆模块的全部证据与原始输入，生成最终回答$\boldsymbol{r}_{\text{ReAct}}$。
$$\boldsymbol{r}_{\text{ReAct}} = \Phi_{\text{sum}}(\mathcal{M}, \boldsymbol{q}, \boldsymbol{v})$$

**范式对比**
两种范式的组件均由定制提示词驱动，各有优劣：
- **PE范式**：规划与执行效率高，但固定流程在复杂场景中可能损失精度；
- **ReAct范式**：通过动态规划适配中间输出，灵活性更强，但迭代机制会降低效率。


<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/architecture.png" />
    <span style=" font-size: 12px; color: black;"><strong>图3</strong>：SpatialAgent 的架构和工作流程。(a) SpatialAgent 中的专业空间感知工具；(b) 用于任务分解和逐步执行的 Plan-Execute 范式；(c) 用于迭代交互和策略优化的 ReAct 范式。</span>
</div>

### 3.4 工具箱

如图3(c)所示，**SpatialAgent**集成了一个包含12种专用空间感知工具的完备工具箱（$\mathcal{T}$），工具分为四大类：通用感知、运动与变换、位姿与几何、辅助工具。每种工具均明确了功能定义、输入输出格式及使用示例。值得注意的是，该工具箱**完全基于开源模型构建**，确保实验可复现性，并能随底层工具的迭代持续优化。

1.  **通用感知**
    集成开放式视觉感知模型：采用Rex-Omni [32] 实现目标计数与精准框定位，其检测结果可作为SAM2 [64] 分割工具的视觉提示，优化目标定位精度并量化目标占比；同时引入3D目标检测工具DetAny3D [102] 提取3D边界框。

2.  **运动与变换**
    针对多帧及视频动态分析需求：集成RAFT [72] 进行稠密光流估计，结合通用感知模块实现相机运动分析与目标跟踪；利用VGGT [75] 输出序列帧相机外参，并通过SIFT [50] 完成鲁棒特征匹配与单应性矩阵估计，支撑几何变换任务。

3.  **位姿与几何**
    核心功能为空间参数与结构推理：通过VGGT [75] 从单帧/多帧输入中估计相机内外参；采用Depth-Anything-V2 [93] 结合室内/室外域专用模型输出度量深度，与通用感知模块联动获取目标或区域深度信息；借助OrientAnything [84] 估计3D目标朝向，实现细粒度视角关系推理；利用MapAnything [34] 完成3D场景重建，并预测点间真实物理距离。

4.  **辅助工具**
    配置通用工具链以支持工具交互与协同调度，其中专用的“终止”动作负责整合工具输出并标记推理完成；同时通过针对性提示工程，增强以开源多模态大语言模型（如Qwen3-VL [4]）为核心的智能体的分步推理能力。
